In [6]:
##RAG piplines - data ingstion to vector db
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\hasrr\AppData\Local\Temp\ipykernel_22072\4146122784.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [7]:
#read all pdfs
def get_data(path):
    d = os.listdir(path)
    all_doc = []
    print(f"total pdf files : {len(d)}")
    for i in d:
        try:
            if i.endswith(".pdf"):
                print(f"reading {i}")
                pdf_load = PyMuPDFLoader(os.path.join(path,i))
                doc = pdf_load.load()
                for t in doc:
                    t.metadata["source_file"] = os.path.join(path,i)
                    t.metadata["file_type"] = "pdf"
                print(f"loaded {len(doc)} pages")
                all_doc.extend(doc)

        except Exception as e:
            print(f"error:{e}")
    print(f"total doc loaded: {len(all_doc)}")
    return all_doc

all_pdf_docs = get_data("../data/pdf")



total pdf files : 2
reading attention.pdf
loaded 11 pages
reading diffRNN.pdf
loaded 9 pages
total doc loaded: 20


In [8]:
all_pdf_docs[0].metadata['file_type']


'pdf'

In [9]:
def split_docs(docs,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,length_function=len , separators=["\n\n","\n"," ",""])
    split_docs = text_splitter.split_documents(docs)
    print(f"split {len(docs)} documents into {len(split_docs)} chunks")
    if split_docs:
        print("example chunk")
        print(f"content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")
    return split_docs

chunks = split_docs(all_pdf_docs)

split 20 documents into 77 chunks
example chunk
content : Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata : {'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf\\attention.pdf', 'file_path': '../data/pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': '../data/pdf\\attention.pdf', 'file_type': 'pdf'}


### embedding and vectorStoreDB

In [2]:
import numpy as np
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer


c:\Users\hasrr\Desktop\DS&ML\TradRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

c:\Users\hasrr\Desktop\DS&ML\TradRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hasrr\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3574.85it/s]


[[ 6.76569045e-02  6.34959415e-02  4.87130694e-02  7.93049410e-02
   3.74480300e-02  2.65279389e-03  3.93749736e-02 -7.09845126e-03
   5.93614578e-02  3.15370075e-02  6.00980893e-02 -5.29051982e-02
   4.06068079e-02 -2.59308275e-02  2.98428014e-02  1.12690229e-03
   7.35149086e-02 -5.03818654e-02 -1.22386590e-01  2.37028617e-02
   2.97265667e-02  4.24768180e-02  2.56337505e-02  1.99516583e-03
  -5.69190606e-02 -2.71598510e-02 -3.29035223e-02  6.60248846e-02
   1.19007163e-01 -4.58791703e-02 -7.26214275e-02 -3.25840227e-02
   5.23413271e-02  4.50552963e-02  8.25299788e-03  3.67023982e-02
  -1.39415748e-02  6.53918609e-02 -2.64272112e-02  2.06383236e-04
  -1.36643667e-02 -3.62810865e-02 -1.95044372e-02 -2.89738085e-02
   3.94270569e-02 -8.84090737e-02  2.62426562e-03  1.36713833e-02
   4.83062342e-02 -3.11566107e-02 -1.17329173e-01 -5.11690341e-02
  -8.85287970e-02 -2.18963120e-02  1.42986383e-02  4.44167443e-02
  -1.34815695e-02  7.43392035e-02  2.66382992e-02 -1.98762938e-02
   1.79191

In [18]:
embeddings.shape

(2, 384)

In [3]:
from typing import List

In [4]:
class EmbeddingManagaer:
    """Handles docs emedding gen using sentenceTransformer"""
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
        model_name : HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"{self.model_name} model loaded successfully")
        except Exception as e:
            print(f"error:{e}")
    def generate_embeddings(self,text: List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("model not loaded")
        print(f"gen embeddings for {len(text)} texts....")
        embedding = self.model.encode(text, show_progress_bar = True)
        print(f"gen embedding with shape {embedding.shape}")
        return embedding

embedding_manager = EmbeddingManagaer()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1331.49it/s]


all-MiniLM-L6-v2 model loaded successfully


### vectorStore

In [10]:
import uuid

In [14]:
class vectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self,collection_name :str = "pdf_documents" ,persist_dir:str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_dir = persist_dir
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_dir , exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_dir)

            self.collection = self.client.get_or_create_collection(name = self.collection_name,metadata={"description":"PDF doc embedding for RAG"})
            print(f"vector store initialized. Collection:{self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error: {e}")
            raise
    def add_document(self,documents:List,embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("no. of docs must match no. of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #preparing data for vectordb
        ids = []
        metadatas = []
        document_text = []
        embedding_list = []
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prep metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content-length"] = len(doc.page_content)
            metadatas.append(metadata)
            #document content
            document_text.append(doc.page_content)

            #embedding
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(ids = ids, embeddings = embedding_list ,metadatas = metadatas , documents = document_text)
            print(f"succsfully added {len(documents)} documents to vecto store")
            print(f"total docs in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error : {e}")
            raise

vectorstore = vectorStore()
vectorstore

vector store initialized. Collection:pdf_documents
existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': '../data/pdf\\attention.pdf', 'file_path': '../data/pdf\\attention.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': '../data/pdf\\attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁuk

In [16]:
#converting the text to embeddings
text = [d.page_content for d in chunks]

#gen embeddings
embeddings = embedding_manager.generate_embeddings(text)

#store it in vector db
vectorstore.add_document(chunks,embeddings)

gen embeddings for 77 texts....


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches: 100%|██████████| 3/3 [00:06<00:00,  2.22s/it]


gen embedding with shape (77, 384)
Adding 77 documents to vector store...
succsfully added 77 documents to vecto store
total docs in collection: 77


In [29]:
vectorstore

### Retriever pipline from vectorstore

In [17]:
from typing import Dict,Any

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self,vectorstore:vectorStore ,embedding_manager: EmbeddingManagaer):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vectorstore
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query:str , top_k: int = 5,score_threshold:float = 1.5) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #gen query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            res = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()],n_results=top_k)

            #process results
            retrieved_docs=[]
            if res['documents'] and res['documents'][0]:
                documents = res['documents'][0]
                metadatas = res['metadatas'][0]
                distances = res['distances'][0]
                ids = res['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    # similarity_score = 1 - distance

                    if distance<=score_threshold:
                        retrieved_docs.append({"id":doc_id,
                                               "content":document,
                                               "metadata":metadata,
                                               "similarity_score":distance,
                                               "distance":distance,
                                               "rank":i+1
                                               })
                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print('no document found')
            return retrieved_docs
        except Exception as e:
            print(f"error:{e}")
            return []

In [37]:
rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [38]:
rag_retriever

In [39]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 1.5
gen embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.36it/s]

gen embedding with shape (1, 384)
retrieved 5 documents (after filtering)


[{'id': 'doc_e87483e4_25',
  'content': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5\nTraining\nThis section describes the training regime for our models.\n5.1\nTraining Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split tokens

In [30]:
vectorstore.collection.count()

77

In [40]:
rag_retriever.retrieve("tell me about RNN.")

Retrieving documents for query: 'tell me about RNN.'
Top K: 5, Score threshold: 1.5
gen embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.24it/s]

gen embedding with shape (1, 384)
retrieved 5 documents (after filtering)


[{'id': 'doc_c876e06d_67',
  'content': 'formed the more traditional tanh-RNN on both of the Ubisoft datasets. The LSTM-RNN was best\nwith the Ubisoft A, and with the Ubisoft B, the GRU-RNN performed best.\nIn Figs. 2–3, we show the learning curves of the best validation runs. In the case of the music\ndatasets (Fig. 2), we see that the GRU-RNN makes faster progress in terms of both the number of\n6',
  'metadata': {'source': '../data/pdf\\diffRNN.pdf',
   'doc_index': 67,
   'trapped': '',
   'source_file': '../data/pdf\\diffRNN.pdf',
   'creationdate': '2014-12-12T01:38:49+00:00',
   'author': '',
   'format': 'PDF 1.5',
   'keywords': '',
   'page': 5,
   'subject': '',
   'creationDate': 'D:20141212013849Z',
   'file_path': '../data/pdf\\diffRNN.pdf',
   'title': '',
   'modDate': 'D:20141212013849Z',
   'total_pages': 9,
   'producer': 'pdfTeX-1.40.12',
   'moddate': '2014-12-12T01:38:49+00:00',
   'file_type': 'pdf',
   'content-length': 357,
   'creator': 'LaTeX with hyperref pa

### INTEGRATION OF VECTOR DB CONTEXT WITH LLM

In [ ]:
##SIMPLE RAG PIPELINE WITH GROQ LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#initialize the groq llm:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

llm = ChatGroq(model="qwen/qwen3-32b",temperature=0.1,max_tokens=1024,reasoning_format='hidden')

#simple RAG function
def rag_simple(llm,retriever,query,top_k=3):
    #retrieve the context
    context = retriever.retrieve(query,top_k)
    context = "\n\n".join([doc['content'] for doc in context]) if context else ""

    #gen ans using groq llm
    prompt = f"""use the following context to answer the question briefly ,
            context : {context}
            question: {query}
            answer:
            """
    response = llm.invoke([prompt.format(context,query=query)])
    return response.content

In [53]:
answer = rag_simple(llm,rag_retriever,"what's attention mechanism")

Retrieving documents for query: 'what's attention mechanism'
Top K: 3, Score threshold: 1.5
gen embeddings for 1 texts....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.42it/s]

gen embedding with shape (1, 384)
retrieved 3 documents (after filtering)


In [54]:
print(answer)

The attention mechanism is a technique that allows models to focus on relevant parts of input data by computing relationships between different positions. In the context of the Transformer, **self-attention** (or intra-attention) relates positions within a single sequence, while **multi-head attention** applies multiple parallel attention functions with different learned projections to capture diverse representation subspaces. This enables the model to jointly attend to information from different positions, improving interpretability and handling long-range dependencies efficiently compared to sequential models like RNNs.


In [58]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3-32b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024,
            reasoning_format='hidden'
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

                    Context:
                    {context}

                    Question: {question}

                    Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

                        Question: {query}

                        Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [62]:
groqllm = GroqLLM()
query = "what is attention mechanism?"
context = rag_retriever.retrieve(query=query)
# context = RAGRetriever.retrieve(query=query)
context = "\n\n".join([doc['content'] for doc in context]) if context else ""
response = groqllm.generate_response(query=query,context=context)

Initialized Groq LLM with model: qwen/qwen3-32b
Retrieving documents for query: 'what is attention mechanism?'
Top K: 5, Score threshold: 1.5
gen embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.74it/s]

gen embedding with shape (1, 384)
retrieved 5 documents (after filtering)


In [63]:
print(response)

The **attention mechanism** is a technique that allows models to dynamically focus on relevant parts of an input sequence when processing each element. It computes relationships between different positions in the sequence, enabling the model to capture dependencies regardless of their distance. 

In the context provided:  
- **Self-attention** (also called *intra-attention*) relates different positions within a single sequence to compute its representation. This helps models understand contextual relationships (e.g., syntactic/semantic structures).  
- **Multi-head attention** enhances this by projecting queries, keys, and values multiple times with different learned parameters, allowing the model to jointly attend to information from diverse representation subspaces. This parallelization improves performance and interpretability, as different attention heads can specialize in distinct tasks.  

The Transformer model relies entirely on self-attention (without recurrent or convolutional

### Enhanced RAG Pipeline Features

In [81]:
def rag_enhanced(query,retriever,llm,top_k=5,score_threshold=1.5,return_context=False):
    try:
        result = retriever.retrieve(query,score_threshold = score_threshold)
    except Exception as e:
        print(f"error while getting context:{e}" )
    if result:
        context = "\n\n".join([d['content'] for d in result])
        prompt = f"provide a concise answer on, \ncontext: {context} , \nbased on the query :{query} "
        message = [HumanMessage(content= prompt)] # <---- it should be a list
        response = llm.invoke(message)
        sources = [{'source':doc['metadata'].get('source_file',doc['metadata'].get("source","unkown")),
                    'page': doc['metadata'].get("page","unkown"),
                    'score': doc['similarity_score'],
                    'preview': doc['content'][:200]+"..."} for doc in result]
        confidence = max([doc['similarity_score'] for doc in result])
        output = {'answer' : response.content , 'sources' : sources , 'confidence_score': confidence}
        if return_context:
            output['context']=context
        return output

In [82]:
query = "what's attention mechanism?"
result = rag_enhanced(query=query,retriever=rag_retriever,llm=llm,return_context=True)

Retrieving documents for query: 'what's attention mechanism?'
Top K: 5, Score threshold: 1.5
gen embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.52it/s]

gen embedding with shape (1, 384)
retrieved 5 documents (after filtering)


In [85]:
print(result['answer'])

The **attention mechanism** is a technique that computes relationships between elements in a sequence by dynamically weighting their interactions. In the Transformer model, it operates as follows:

1. **Core Components**:  
   - **Queries (Q), Keys (K), Values (V)**: These are derived from input embeddings via learned linear projections.  
   - **Dot-Product Attention**: Computes compatibility between queries and keys as $ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $, where $ d_k $ is the key dimension. The scaling factor $ \frac{1}{\sqrt{d_k}} $ prevents large dot products from destabilizing the softmax gradients.

2. **Multi-Head Attention**:  
   - Projects Q, K, V into $ h $ different subspaces using distinct learned linear transformations.  
   - Performs attention independently in each subspace, concatenates results, and applies a final linear projection.  
   - Allows the model to jointly attend to information from different representation su

### advanced rag pipline

In [94]:
from typing import List,Dict,Any
import time
class ADVANCEDRAGPipeline:
    def __init__(self,retriever = rag_retriever,llm = llm):
        self.retriever = retriever 
        self.llm = llm
        self.history = []
    def AdvRag(self,query,top_k=5,score_threshold=1.5,stream=False,summarize = False):
        try:
            result = self.retriever.retrieve(query=query,score_threshold=score_threshold)
        except Exception as e:
            print(f"couldn't load context : {e}")
        if result:
            context = "\n\n".join([doc['content'] for doc in result])
            prompt = f"you are a helpful agent can you give concise answer,\non given context:{context},\n for given query:{query}"
            msg = [HumanMessage(content=prompt)]
            response = self.llm.invoke(msg)
            #just stimulating
            if stream:
                for i in range(0,len(prompt),80):
                    print(prompt[i:i+80],end="" ,flush=True)
                    time.sleep(0.05)
                print()
            if summarize:
                sum = f"can you summarize the context:{context} based on query:{query} in 3 sentences."
                msg = [HumanMessage(content=sum)]
                summary = self.llm.invoke(msg)
                summary = summary.content
            sources = [{'source':doc['metadata'].get('source_file',doc['metadata'].get("source","unkown")),
                    'page': doc['metadata'].get("page","unkown"),
                    'score': doc['similarity_score'],
                    'preview': doc['content'][:200]+"..."} for doc in result]
            
            confidence = max([doc['similarity_score'] for doc in result])
            output = {'question':query ,'answer' : response.content , 'sources' : sources , 'confidence_score': confidence, 'summary' : summary}
            self.history.append(output)
            return output

In [95]:
rag = ADVANCEDRAGPipeline(retriever=rag_retriever,llm=llm)

In [96]:
query = "what is RNN?"
response = rag.AdvRag(query=query , stream=True , summarize=True)

Retrieving documents for query: 'what is RNN?'
Top K: 5, Score threshold: 1.5
gen embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.04it/s]

gen embedding with shape (1, 384)
retrieved 5 documents (after filtering)


you are a helpful agent can you give concise answer,
on given context:formed the more traditional tanh-RNN on both of the Ubisoft datasets. The LSTM-RNN was best
with the Ubisoft A, and with the Ubisoft B, the GRU-RNN performed best.
In Figs. 2–3, we show the learning curves of the best validation runs. In the case of the music
datasets (Fig. 2), we see that the GRU-RNN makes faster progress in terms of both the number of
6

sequence by having a recurrent hidden state whose activation at each time is dependent on that of
the previous time.
More formally, given a sequence x = (x1, x2, · · · , xT), the RNN updates its recurrent hidden state
ht by
ht =
0,
t = 0
φ (ht−1, xt) ,
otherwise
(1)
where φ is a nonlinear function such as composition of a logistic sigmoid with an afﬁne transforma-
tion. Optionally, the RNN may have an output y = (y1, y2, . . . , yT) which may again be of variable
length.
Traditionally, the update of the recurrent hidden state in Eq. (1) is implemented as
ht = g (W

In [97]:
rag.history

[{'question': 'what is RNN?',
  'answer': 'An **RNN (Recurrent Neural Network)** is a type of neural network designed to process sequential data by maintaining a hidden state that captures information from previous time steps. It updates its hidden state $ h_t $ using the current input $ x_t $ and the previous hidden state $ h_{t-1} $, typically via a nonlinear function like $ h_t = \\phi(Wx_t + Uh_{t-1}) $. RNNs model sequences by generating probability distributions over next elements, enabling variable-length sequence generation (e.g., ending with a special symbol). However, traditional RNNs struggle with long-term dependencies due to vanishing/exploding gradients, motivating variants like LSTM and GRU.',
  'sources': [{'source': '../data/pdf\\diffRNN.pdf',
    'page': 5,
    'score': 1.107006311416626,
    'preview': 'formed the more traditional tanh-RNN on both of the Ubisoft datasets. The LSTM-RNN was best\nwith the Ubisoft A, and with the Ubisoft B, the GRU-RNN performed best.\n

In [93]:
response['summary']

'An RNN (Recurrent Neural Network) is a type of neural network designed to process sequential data by maintaining a hidden state that captures information from previous time steps, allowing it to model temporal dependencies. It updates its hidden state using a nonlinear function (e.g., tanh or sigmoid) that combines the current input with the previous hidden state, enabling it to generate outputs or predict sequences. Variants like LSTM and GRU improve upon traditional RNNs by addressing challenges like vanishing gradients, making them better suited for capturing long-term dependencies in tasks such as music or text generation.'